What is total revenue for quarter 1996-Q1, and how does it compare to the previous
quarter?


In [0]:
%sql
SELECT 
    quarter_start, 
    SUM(net_revenue) AS total_revenue
FROM workspace.gold.revenue_metrics
WHERE quarter_start IN ('1995-10-01', '1996-01-01')
GROUP BY quarter_start
ORDER BY quarter_start;

How many active customers (at least one order in the last year of data) were there as of
1998-08-02?


In [0]:
%sql
SELECT COUNT(c_custkey) AS active_customers_count
FROM workspace.gold.customer_activity
WHERE is_active_trailing_12m = TRUE;

If revenue dropped 20% week over week, which alert rule would catch it? Show the query
and demonstrate it against artificially reduced data

In [0]:
%sql
WITH weekly_revenue AS (
    SELECT 
        week_start, 
        SUM(net_revenue) AS current_rev
    FROM workspace.gold.revenue_metrics
    GROUP BY week_start
)
SELECT 
    w1.week_start AS current_week,
    w1.current_rev,
    w2.current_rev AS previous_rev,
    ((w1.current_rev - w2.current_rev) / w2.current_rev) * 100 AS drop_percentage
FROM weekly_revenue w1
JOIN weekly_revenue w2 
    ON w1.week_start = date_add(w2.week_start, 7)
WHERE ((w1.current_rev - w2.current_rev) / w2.current_rev) <= -0.20
ORDER BY w1.week_start;

In [0]:
%sql
--Створюємо тимчасовий набір даних, де для одного з тижнів штучно обрізаємо дохід на 50%
WITH artificially_reduced_data AS (
    SELECT 
        week_start,
        CASE 
            WHEN week_start = '1998-05-04' THEN SUM(net_revenue) * 0.5 -- штучне падіння на 50%
            ELSE SUM(net_revenue)
        END AS net_revenue
    FROM workspace.gold.revenue_metrics
    GROUP BY week_start
),

weekly_revenue AS (
    SELECT 
        week_start, 
        net_revenue AS current_rev
    FROM artificially_reduced_data
)
SELECT 
    w1.week_start AS current_week,
    w1.current_rev,
    w2.current_rev AS previous_rev,
    ((w1.current_rev - w2.current_rev) / w2.current_rev) * 100 AS drop_percentage
FROM weekly_revenue w1
JOIN weekly_revenue w2 
    ON w1.week_start = date_add(w2.week_start, 7)
WHERE ((w1.current_rev - w2.current_rev) / w2.current_rev) <= -0.20
ORDER BY w1.week_start;